# Data Challenge 13 — Interpreting Logistic Regression 

**Purpose**  
Apply what you learned about logistic regression interpretation by analyzing NYC Restaurant Inspection data. 
 
You’ll practice interpreting **continuous**, **binary**, and **categorical** predictors, compute **odds ratios**, and assess model accuracy. 

**Learning Goals**
- Convert coefficients to odds ratios using `np.exp()`.  
- Interpret ORs for continuous, binary, and categorical predictors.  
- Use accuracy to assess logistic regression performance.  
- Communicate results clearly and responsibly.  

**Data:** June 1, 2025 - Nov 4, 2025 Restaurant Health Inspection

[Restaurant Health Inspection](https://data.cityofnewyork.us/Health/DOHMH-New-York-City-Restaurant-Inspection-Results/43nn-pn8j/about_data)


## Instructor Guidance

**Hint: Use the Lecture Deck, Canvas Reading, and Docs to help you with the code**

Use this guide live; students implement below.

**Docs (Quick Links)**
- LogisticRegression — https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html  
- accuracy_score — https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html  
- OneHotEncoder — https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html  
- StandardScaler — https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html  
- np.exp — https://numpy.org/doc/stable/reference/generated/numpy.exp.html  

**Pseudocode Plan**

1️⃣ Load cleaned restaurant inspection data from the previous challenge.  
2️⃣ Define target = `IS_A` (1 = Grade A, 0 = otherwise).  
3️⃣ Predictors →  
    • Continuous = `SCORE`  
    • Binary = `CRITICAL_NUM`  
    • Categorical = `BORO`  
4️⃣ Scale continuous variables; encode categorical ones.  
5️⃣ Fit `LogisticRegression`.  
6️⃣ Exponentiate coefficients (np.exp()) → odds ratios.  
7️⃣ Interpret one continuous, one binary, and one categorical coefficient.  
8️⃣ Evaluate accuracy.  
9️⃣ Reflect on scaling choices and communication of odds.  


## You Do — Student Section
Work in pairs. Comment your choices briefly. Keep code simple—only coerce the columns you use.

## Step 1 — Imports and Plot Defaults

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


### Step 2 — Load CSV, Create Columns, Preview

- Point to your New York City Restaurant Inspection Data 
- Create the `is_A` and `critical_num` columns like you did in L11 notebook

In [2]:
# Set the option to display all columns
pd.set_option('display.max_columns', None)
df = pd.read_csv('/Users/Marcy_Student/Desktop/marcy/DA2025_Lectures_Kevin/Mod6/data/DOHMH_New_York_City_Restaurant_Inspection_Results_20251110.csv',low_memory=False)

# turns each column name into lower case
df.columns = df.columns.str.lower()


df['score'] = pd.to_numeric(df['score'], errors='coerce')
df['grade'] = df['grade'].astype(str).str.upper()
df['critical flag'] = df['critical flag'].astype(str).str.lower()

# if grade is A then 1 else 0
df['is_A'] = (df['grade'] == 'A').astype(int)
# if critical flag is critical then 1 else 0
df['critical_num'] = (df['critical flag'] == 'critical').astype(int)

# drops any NA
df = df.dropna(subset=['score', 'is_A','boro']).copy()
df.head()


,camis,dba,boro,building,street,zipcode,phone,cuisine description,inspection date,action,violation code,violation description,critical flag,score,grade,grade date,record date,inspection type,latitude,longitude,community board,council district,census tract,bin,bbl,nta,location,is_A,critical_num
5,50164779,IDLEWILD CHOP HOUSE,Queens,NaN,TERMINAL 8 JFK INTL,NaN,7186790452,American,06/25/2025,Violations were cited in the following area(s).,02B,Hot TCS food item not held at or above 140 °F.,critical,13.0,A,06/25/2025,11/10/2025,Pre-permit (Operational) / Initial Inspection,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1
15,50062738,CITI FIELD BUD BAR STAND 336,Queens,126TH ST,AND ROOSEVELT AVE,NaN,7185958192,Other,09/20/2024,Violations were cited in the following area(s).,04A,Food Protection Certificate (FPC) not held by ...,critical,10.0,A,09/20/2024,11/10/2025,Cycle Inspection / Initial Inspection,0.000000,0.000000,NaN,NaN,NaN,NaN,4.000000e+00,NaN,NaN,1,1
20,41406895,SUN SAI GAI RESTAURANT,Manhattan,220,CANAL STREET,10013.0,6463987353,Chinese,09/22/2025,Establishment re-opened by DOHMH.,NaN,NaN,not applicable,0.0,Z,09/22/2025,11/10/2025,Cycle Inspection / Reopening Inspection,40.717139,-73.998795,103.0,1.0,2900.0,1079404.0,1.001990e+09,MN27,POINT (-73.998795131761 40.717138793492),0,0
27,40942152,CROTON RESERVOIR TAVERN,Manhattan,108110,WEST 40 STREET,NaN,2129976835,American,02/01/2022,Violations were cited in the following area(s).,02B,Hot food item not held at or above 140º F.,critical,9.0,A,02/01/2022,11/10/2025,Cycle Inspection / Initial Inspection,0.000000,0.000000,NaN,NaN,NaN,NaN,1.000000e+00,NaN,NaN,1,1
36,50066129,GYRO CAFE,Brooklyn,580,CONEY ISLAND AVENUE,11218.0,7184380860,Pakistani,12/07/2023,Establishment re-opened by DOHMH.,NaN,NaN,not applicable,0.0,NAN,NaN,11/10/2025,Cycle Inspection / Reopening Inspection,40.643213,-73.969948,312.0,40.0,49200.0,3125633.0,3.053610e+09,BK41,POINT (-73.969947828546 40.643212575642),0,0


## Step 3 — Define Predictors & Target

- Target is `is_A` 
- X predictors are: SCORE, CRITICAL_NUM (created in Step 2), BORO


In [3]:
Y = df['is_A']
X = df[['score', 'critical_num','boro']]

## Step 4 — Split Data (70/30 Stratify by Target)

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.30, stratify=Y, random_state=42
)
print("Train size:", X_train.shape[0], "Test size:", X_test.shape[0])

Train size: 192968 Test size: 82701


## Step 5 – Preprocessing (You can chose to do this in a Pipeline)  

- Scale continuous features  
- Pass binary as is  
- One-hot encode categorical feature (`BORO`)  

In [20]:
# defining preprocessing for columns that are being transformed
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


preprocessor = ColumnTransformer(
        transformers=[
            ('categorical', OneHotEncoder(handle_unknown='ignore',drop='first'),['boro'])
        ],
        # ignoring columns that were not referenced in preprocessing
        remainder='passthrough'
)
# defining pipeline
model = Pipeline(
    steps=
        [
        # using predefined processing for model
        ('processing',preprocessor),
        # regression that would be performed
        ("model", LogisticRegression())
        ]
)

## Step 6 – Fit Model & Evaluate Accuracy

- Fit `is_A ~ score` using **LogisticRegression**  
- Compute predictions with `.predict()`  
- Evaluate accuracy with `accuracy_score()`

In [21]:
model.fit(X_train,y_train)

predictions = model.predict(X_test)

print('The Accuracy Score is:',round(accuracy_score(y_test, (predictions >= 0.5).astype(int)),5))

The Accuracy Score is: 0.95177


/opt/miniconda3/lib/python3.12/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/opt/miniconda3/lib/python3.12/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/opt/miniconda3/lib/python3.12/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/opt/miniconda3/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/opt/miniconda3/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/opt/miniconda3/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


## Step 7 – Extract Coefficients and Convert to Odds Ratios


In [22]:

names = preprocessor.get_feature_names_out()
coefficients = model.named_steps['model'].coef_[0]
oddsratio = np.exp(coefficients)

coef = pd.DataFrame({
    'columns':names,
    'coefficients':coefficients,
    'oddsratio':oddsratio
})
coef


,columns,coefficients,oddsratio
0,categorical__boro_Brooklyn,0.028167,1.028568
1,categorical__boro_Manhattan,0.052938,1.054364
2,categorical__boro_Queens,0.128886,1.137561
3,categorical__boro_Staten Island,0.099968,1.105136
4,remainder__score,-0.327929,0.720414
5,remainder__critical_num,-0.084806,0.918690


## Step 8 – Interpret Each Predictor 

**Remember**
💡 OR > 1 → increases odds of Grade A  
💡 OR < 1 → decreases odds of Grade A

Here are three clear interpretation statements based on the odds ratios:

- Restaurants with a **higher inspection score** are **less likely** to receive an A, since the odds ratio is **0.72**, meaning each additional point on the score lowers the odds of getting an A by about **28%**.
- Being located in **Queens** slightly **increases the odds** of getting an A, with an odds ratio of **1.14**, or about a **14% increase** compared to the baseline borough.
- The number of **critical violations** also decreases the chance of earning an A, with an odds ratio of **0.92**, meaning each additional critical violation lowers the odds by about **8%**.



# We Share — Reflection & Wrap-Up

Write **one short paragraphs** (4–6 sentences). Be specific and use evidence from your notebook.
"Compare the 3 models, why do odd ratios make more sense for stakeholders"
**Which predictor had the strongest relationship with getting an A grade?**  
Use the odds ratios and accuracy to support your answer.  
- When comparing the three models, the last one made the most sense because it was accurate but also easy to explain. Using odds ratios helped the most, since they translate the math into everyday language (like saying something increases or decreases someone’s chances) which is much clearer for restaurant owners or city staff. From the results, the inspection score had the strongest impact: for every point a restaurant’s score went up, its chances of getting an A went down by about 28%, which makes sense since higher scores mean more violations. Location also mattered — for example, restaurants in Queens had slightly better odds of earning an A compared to the baseline borough. Overall, this model helps people understand not just the prediction, but why it happens.